# 🇮🇳 GeM Scraper — Full RAG Pipeline on Google Colab

Runs **exactly like local** — same commands, same data, same results.

```
python main.py --ask "Healthcare and Medical tenders"
python main.py --ask "tenders in Tamil Nadu"
python main.py --tender-file active_tenders-mittal-ind.json
python main.py --reindex
python main.py --stats
```

**Run cells 1 → 5 once to set up, then Cell 6+ for queries.**


## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted ✓')

## Cell 2 — Clone Project from GitHub

In [ ]:
import os

GITHUB_TOKEN = ''           # ← paste your PAT token here if repo is private
GITHUB_USER  = 'Darshan3123'
REPO_NAME    = 'RAG'
BRANCH       = 'colab'
PROJECT_DIR  = '/content/gem_scraper'

if GITHUB_TOKEN:
    REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
else:
    REPO_URL = f'https://github.com/{GITHUB_USER}/{REPO_NAME}.git'

if not os.path.exists(PROJECT_DIR):
    ret = os.system(f'git clone --branch {BRANCH} "{REPO_URL}" {PROJECT_DIR}')
    if ret != 0 or not os.path.exists(PROJECT_DIR):
        raise RuntimeError('Clone failed — check GITHUB_TOKEN or make repo public')
    print(f'Cloned ✓  →  {PROJECT_DIR}')
else:
    os.system(f'git -C {PROJECT_DIR} pull')
    print(f'Already exists, pulled latest')

os.chdir(PROJECT_DIR)
print(f'Working dir: {os.getcwd()}')
print('Project files:', [f for f in os.listdir('.') if not f.startswith('.')])

## Cell 3 — Install Dependencies

In [ ]:
import subprocess, sys

# Install RAG dependencies (Playwright/scraper not needed in Colab)
packages = [
    'python-dotenv',
    'sentence-transformers',
    'rank-bm25',
    'chromadb',
    'openai',
    'requests',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)

# Check GPU
import torch
print(f'\nCUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('Running on CPU (RAG still works, just slower)')
print('Dependencies installed ✓')

## Cell 4 — Configure Paths

Set `DRIVE_DATA` to your Google Drive folder.
Upload `gem_bids.db` and `chroma_db/` there from your Windows machine.

In [ ]:
import os, sys

PROJECT_DIR = '/content/gem_scraper'
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

# ── Google Drive storage folder ───────────────────────────────────────
# Upload gem_bids.db and chroma_db/ here from your Windows machine:
#   D:\Projects\gem_scraper\storage\gem_bids.db  →  Google Drive/gem_scraper_data/
#   D:\Projects\gem_scraper\storage\chroma_db\  →  Google Drive/gem_scraper_data/
DRIVE_DATA = '/content/drive/MyDrive/gem_scraper_data'
os.makedirs(DRIVE_DATA, exist_ok=True)

# ── Set env vars (mirrors .env file) ──────────────────────────────────
os.environ['DB_PATH']           = f'{DRIVE_DATA}/gem_bids.db'
os.environ['CHROMA_DIR']        = f'{DRIVE_DATA}/chroma_db'
os.environ['JSON_OUT_PATH']     = f'{DRIVE_DATA}/gem_bids.json'
os.environ['TENDER_DB_PATH']    = f'{DRIVE_DATA}/tenders.db'
os.environ['TENDER_JSON_PATH']  = f'{DRIVE_DATA}/tenders.json'
os.environ['LOG_DIR']           = f'{PROJECT_DIR}/logs'
os.environ['DOWNLOAD_DIR']      = f'{PROJECT_DIR}/downloads'

os.environ['RAG_EMBEDDING_MODEL'] = 'BAAI/bge-base-en-v1.5'
os.environ['RAG_RERANKER_MODEL']  = 'BAAI/bge-reranker-base'
os.environ['RAG_USE_RERANKER']    = 'true'
os.environ['RAG_TOP_K']           = '5'
os.environ['RAG_FETCH_K']         = '40'
os.environ['HF_OFFLINE']          = 'false'
os.environ['RAG_LLM_PROVIDER']    = ''     # retrieval-only (no LLM needed)
os.environ['TESSERACT_CMD']       = 'tesseract'

for d in [DRIVE_DATA, f'{PROJECT_DIR}/logs', f'{PROJECT_DIR}/downloads',
          f'{PROJECT_DIR}/storage']:
    os.makedirs(d, exist_ok=True)

# Symlink storage → Drive so code finds files at storage/gem_bids.db
LOCAL_STORAGE = f'{PROJECT_DIR}/storage'
for fname in ['gem_bids.db', 'chroma_db', 'gem_bids.json', 'tenders.db']:
    src = f'{DRIVE_DATA}/{fname}'
    dst = f'{LOCAL_STORAGE}/{fname}'
    if os.path.exists(src) and not os.path.exists(dst):
        os.symlink(src, dst)
        print(f'  symlink: storage/{fname} → Drive')

print('\nConfiguration:')
print(f'  DB        : {os.environ["DB_PATH"]}')
print(f'  ChromaDB  : {os.environ["CHROMA_DIR"]}')
print(f'  Embedding : {os.environ["RAG_EMBEDDING_MODEL"]}')
print(f'  LLM       : {os.environ.get("RAG_LLM_PROVIDER") or "retrieval-only"}')

# Check if DB exists
db_path = os.environ['DB_PATH']
if os.path.exists(db_path):
    size = os.path.getsize(db_path) / 1024
    print(f'\n  gem_bids.db  : FOUND ({size:.0f} KB) ✓')
else:
    print(f'\n  gem_bids.db  : NOT FOUND')
    print('  → Upload from Windows: D:\\Projects\\gem_scraper\\storage\\gem_bids.db')
    print('    to Google Drive folder: gem_scraper_data/')

chroma_path = os.environ['CHROMA_DIR']
if os.path.exists(chroma_path):
    print(f'  chroma_db    : FOUND ✓')
else:
    print(f'  chroma_db    : NOT FOUND (will rebuild with --reindex)')

## Cell 5 — Load Data (if gem_bids.db not on Drive)

**Option A:** Load from tender JSON file (already in repo)

**Option B:** Upload `gem_bids.db` from Windows to Drive (faster, keeps all data)

In [ ]:
import os, json, sys
sys.path.insert(0, '/content/gem_scraper')
os.chdir('/content/gem_scraper')

from pipeline.tender_parser import parse_api_response
from storage.database import BidDatabase

db = BidDatabase()
s  = db.stats()

if s['total'] > 0:
    print(f'Database already has {s["total"]} bids — skipping load')
else:
    # Load from JSON files in the repo
    json_files = [
        'active_tenders-mittal-ind.json',
        'tender_result.json',
    ]
    total_loaded = 0
    for jf in json_files:
        path = f'/content/gem_scraper/{jf}'
        if not os.path.exists(path):
            print(f'  Skipping (not found): {jf}')
            continue
        with open(path, encoding='utf-8') as f:
            raw = json.load(f)
        records = parse_api_response(raw)
        print(f'  Parsed {len(records)} records from {jf}')
        for rec in records:
            bid = {
                'bid_no':           rec.get('tender_no', '') or rec.get('tender_reference_id', ''),
                'document_url':     rec.get('source_url', '') or rec.get('tender_no', ''),
                'full_item_name':   rec.get('category', '') or rec.get('search_text', '')[:200],
                'department':       rec.get('authority', ''),
                'authority':        rec.get('authority', ''),
                'sector':           rec.get('sector', ''),
                'state':            rec.get('state', ''),
                'city':             rec.get('city', ''),
                'status':           rec.get('status', 'OPEN'),
                'tender_status':    rec.get('status', 'OPEN'),
                'procurement_type': rec.get('procurement_type', ''),
                'tender_type':      rec.get('tender_type', ''),
                'bid_type':         rec.get('tender_type', ''),
                'product_type':     rec.get('product_name', ''),
                'category':         rec.get('category', ''),
                'sub_category':     rec.get('sub_category', ''),
                'end_date':         rec.get('due_date', ''),
                'start_date':       rec.get('pub_date', ''),
                'tender_value':     rec.get('tender_value', ''),
                'estimated_value':  rec.get('tender_value', ''),
                'work_desc':        rec.get('work_desc', ''),
                'tender_summary':   rec.get('tender_summary', ''),
                'full_pdf_text':    (rec.get('work_desc', '') or rec.get('tender_summary', '')),
                'address':          rec.get('address', ''),
                'address_pin':      rec.get('address_pin', ''),
                'contact_person':   rec.get('contact_person', ''),
                'contact_email':    rec.get('contact_email', ''),
                'contact_phone':    rec.get('contact_phone', ''),
                'procurement_source': rec.get('source_url', ''),
            }
            if not bid['document_url']:
                continue
            db.upsert(bid)
            total_loaded += 1
    print(f'\nLoaded {total_loaded} bids into gem_bids.db ✓')

s = db.stats()
print(f'Total bids in DB: {s["total"]}')

## Cell 6 — Build Vector Store Index

Same as: `python main.py --reindex`

In [ ]:
import sys, os
sys.path.insert(0, '/content/gem_scraper')
os.chdir('/content/gem_scraper')

from storage.database import BidDatabase
from rag.vector_store import reindex_all, stats as vs_stats

db = BidDatabase()
vs = vs_stats()

print(f'SQLite bids    : {db.stats()["total"]}')
print(f'ChromaDB chunks: {vs["total_chunks"]}')

if vs['total_chunks'] == 0:
    print('\nBuilding vector index...')
    reindex_all(db)
    vs = vs_stats()
    print(f'\nIndex built ✓  →  {vs["total_chunks"]} chunks')
else:
    print('\nVector store already has data.')
    print('To force rebuild: reindex_all(db)')

## Cell 7 — Stats

Same as: `python main.py --stats`

In [ ]:
import sys, os
sys.path.insert(0, '/content/gem_scraper')

from storage.database import BidDatabase
from rag.vector_store import stats as vs_stats

db = BidDatabase()
s  = db.stats()
vs = vs_stats()

print('=' * 42)
print('  GeM Bid Scraper — Stats')
print('=' * 42)
print(f'  SQLite total bids  : {s["total"]}')
print(f'  New (unseen)       : {s["new_this_run"]}')
print(f'  Total runs logged  : {s["total_runs"]}')
print(f'  Vector store chunks: {vs["total_chunks"]}')
print(f'  ChromaDB path      : {vs["chroma_dir"]}')
print('=' * 42)

## Cell 8 — RAG Query Engine

Same as: `python main.py --ask "your question"`

In [ ]:
import sys, os
sys.path.insert(0, '/content/gem_scraper')
os.chdir('/content/gem_scraper')

from rag.query_engine import QueryEngine

engine = QueryEngine()

def ask(question, filters=None):
    """Same as: python main.py --ask 'question' [--filter key=value]"""
    result = engine.ask(question, filters=filters)
    print(f"\nQ: {result['question']}")
    if result.get('answer') and result['answer'] != 'No relevant bids found for your query.':
        print(f"A: {result['answer']}")
    print(f"\n── Sources ({len(result['sources'])} unique bids) ──")
    for s in result['sources']:
        sector = s.get('sector', '')
        state  = s.get('state', '')
        status = s.get('status', '')
        meta   = ' | '.join(filter(None, [sector, state, status]))
        print(
            f"  [{s['bid_no']}]  "
            f"{s['department'][:35]}  "
            f"Score: {s['relevance_score']:.0%}"
            + (f"  [{meta}]" if meta else "")
        )
    print()
    return result

print('QueryEngine ready ✓  (auto-filter detection enabled)')
print('Usage: ask("Healthcare and Medical tenders")')

## Cell 9 — Run Queries

Auto-filter detects state/sector/status from the question automatically.

In [ ]:
ask('Healthcare and Medical tenders')

In [ ]:
ask('defence and security tenders')

In [ ]:
ask('tenders in Tamil Nadu')

In [ ]:
ask('OPEN tenders in Chhattisgarh')

In [ ]:
ask('Nuclear and Atomic Energy tenders')

In [ ]:
ask('services tenders in Uttar Pradesh')

In [ ]:
# Explicit filter — same as: --filter sector=Defence and Security
ask('all tenders', filters={'sector': 'Defence and Security'})

In [ ]:
# All OPEN tenders — same as: --filter status=OPEN
ask('all tenders', filters={'status': 'OPEN'})

## Cell 10 — Load Tender File

Same as: `python main.py --tender-file active_tenders-mittal-ind.json`

In [ ]:
import sys, os, json
sys.path.insert(0, '/content/gem_scraper')
os.chdir('/content/gem_scraper')

from pipeline.tender_parser import parse_api_response
from storage.database import BidDatabase
from rag.vector_store import reindex_all

# Change this to any JSON file path
TENDER_FILE = '/content/gem_scraper/active_tenders-mittal-ind.json'

if not os.path.exists(TENDER_FILE):
    print(f'File not found: {TENDER_FILE}')
else:
    db = BidDatabase()
    with open(TENDER_FILE, encoding='utf-8') as f:
        raw = json.load(f)
    records = parse_api_response(raw)
    print(f'Parsed {len(records)} records')

    new_count = 0
    for rec in records:
        bid = {
            'bid_no':           rec.get('tender_no', ''),
            'document_url':     rec.get('source_url', '') or rec.get('tender_no', ''),
            'full_item_name':   rec.get('category', '') or rec.get('search_text', '')[:200],
            'department':       rec.get('authority', ''),
            'authority':        rec.get('authority', ''),
            'sector':           rec.get('sector', ''),
            'state':            rec.get('state', ''),
            'city':             rec.get('city', ''),
            'status':           rec.get('status', 'OPEN'),
            'tender_status':    rec.get('status', 'OPEN'),
            'procurement_type': rec.get('procurement_type', ''),
            'tender_type':      rec.get('tender_type', ''),
            'bid_type':         rec.get('tender_type', ''),
            'product_type':     rec.get('product_name', ''),
            'category':         rec.get('category', ''),
            'sub_category':     rec.get('sub_category', ''),
            'end_date':         rec.get('due_date', ''),
            'start_date':       rec.get('pub_date', ''),
            'estimated_value':  rec.get('tender_value', ''),
            'work_desc':        rec.get('work_desc', ''),
            'tender_summary':   rec.get('tender_summary', ''),
            'full_pdf_text':    rec.get('work_desc', '') or rec.get('tender_summary', ''),
            'address':          rec.get('address', ''),
            'contact_person':   rec.get('contact_person', ''),
            'contact_email':    rec.get('contact_email', ''),
        }
        if not bid['document_url']:
            continue
        is_new = db.upsert(bid)
        if is_new:
            new_count += 1

    s = db.stats()
    print(f'Total in DB: {s["total"]} | New: {new_count}')
    print('\nRebuilding vector index...')
    reindex_all(db)
    print('Done ✓')

## Cell 11 — Tender Stats

Same as: `python main.py --tender-stats`

In [ ]:
import sys
sys.path.insert(0, '/content/gem_scraper')

from storage.tender_database import TenderDatabase

db = TenderDatabase()
s  = db.stats()
print('=' * 42)
print('  Tender Database — Stats')
print('=' * 42)
print(f'  Total tenders : {s["total"]}')
print(f'  OPEN          : {s["open"]}')
print(f'  AOC (results) : {s["aoc"]}')
print(f'  New this run  : {s["new"]}')
print(f'  Total runs    : {s["runs"]}')
print('=' * 42)